# Online Purchase Intention Prediction

## Preprocessing & Modelling

### Purpose of this notebook

This notebook picks up where `01_data_audit_eda.ipynb` left off. It builds a reproducible preprocessing pipeline, establishes baseline models, trains and tunes an XGBoost model, and evaluates performance.

The analysis focuses on:

- Train/test split (stratified, given the ~15.5% class imbalance found in notebook 1)
- Encoding of categorical features
- Dummy Classifier and untuned Random Forest baselines
- Initial XGBoost model with class-imbalance handling
- Model comparison with and without `PageValues` (per the leakage discussion in notebook 1)
- Hyperparameter tuning (Randomized Search → Grid Search → Manual Search)
- Feature importance and business interpretation

### Input

This notebook loads the same raw dataset used in notebook 1 (`../data/raw/online_shoppers_intention.csv`). No rows were removed in notebook 1 (the 125 duplicates were intentionally retained), so the raw file is reloaded here directly.

## 1. Imports & Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

SEED = 1

## 2. Load Data

In [2]:
shopping_df = pd.read_csv("../data/raw/online_shoppers_intention.csv")
shopping_df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 3. Define X / y

In [3]:
X = shopping_df.drop("Revenue", axis=1)
y = shopping_df["Revenue"]

# Confirm class balance (should match notebook 1: ~84.5% / ~15.5%)
y.value_counts(normalize=True)

Revenue
False    0.845255
True     0.154745
Name: proportion, dtype: float64

## 4. Train / Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (9864, 17)
X_test shape:  (2466, 17)


In [5]:
# Verify stratify worked as expected
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Revenue
False    0.845296
True     0.154704
Name: proportion, dtype: float64
Revenue
False    0.845093
True     0.154907
Name: proportion, dtype: float64


## 5. Categorical vs Numerical Columns

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as `int64` but are actually category codes (no meaningful order), so they're converted to `str` before splitting columns by dtype.

In [6]:
cols_to_convert = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']
X_train[cols_to_convert] = X_train[cols_to_convert].astype(str)
X_test[cols_to_convert] = X_test[cols_to_convert].astype(str)

numerical_cols = X_train.select_dtypes(include='number').columns
categorical_cols = X_train.select_dtypes(include='object').columns

print(numerical_cols)
print(categorical_cols)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'],
      dtype='object')
Index(['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
       'VisitorType'],
      dtype='object')


In [7]:
# Check cardinality before encoding
for col in categorical_cols:
    print(col, X_train[col].nunique())

Month 10
OperatingSystems 8
Browser 13
Region 9
TrafficType 20
VisitorType 3


## 6. One-Hot Encoding

In [8]:
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

X_train_cat_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_cat_encoded = encoder.transform(X_test[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_cols, index=X_train.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_cols, index=X_test.index)

X_train_cat_df.head()

,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,OperatingSystems_2,...,TrafficType_20,TrafficType_3,TrafficType_4,TrafficType_5,TrafficType_6,TrafficType_7,TrafficType_8,TrafficType_9,VisitorType_Other,VisitorType_Returning_Visitor
7349,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8611,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3877,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2625,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3508,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 7. Combine Numerical + Encoded Categorical Features

In [9]:
X_train_final = pd.concat([X_train[numerical_cols], X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test[numerical_cols], X_test_cat_df], axis=1)

print(X_train_final.shape)
print(X_test_final.shape)

(9864, 67)
(2466, 67)


## 8a. Baseline Models - Dummy Classifier

In [10]:
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train_final, y_train)
y_pred_dummy = dummy.predict(X_test_final)

print(classification_report(y_test, y_pred_dummy))

              precision    recall  f1-score   support

       False       0.85      1.00      0.92      2084
        True       0.00      0.00      0.00       382

    accuracy                           0.85      2466
   macro avg       0.42      0.50      0.46      2466
weighted avg       0.71      0.85      0.77      2466



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 8b. Baseline Models - Random Forest

In [11]:
rf = RandomForestClassifier(random_state=SEED)
rf.fit(X_train_final, y_train)
y_pred_rf = rf.predict(X_test_final)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

       False       0.92      0.97      0.95      2084
        True       0.78      0.55      0.65       382

    accuracy                           0.91      2466
   macro avg       0.85      0.76      0.80      2466
weighted avg       0.90      0.91      0.90      2466



## 9a. XGBoost — Initial Model

In [12]:
xgb = XGBClassifier(random_state=SEED, eval_metric='logloss')
xgb.fit(X_train_final, y_train)
y_pred_xgb = xgb.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

       False       0.93      0.96      0.94      2084
        True       0.73      0.59      0.65       382

    accuracy                           0.90      2466
   macro avg       0.83      0.77      0.80      2466
weighted avg       0.90      0.90      0.90      2466



## 9b. XGBoost — Handling Class Imbalance (scale_pos_weight)

In [13]:
scale_pos_weight = (y_train == False).sum() / (y_train == True).sum()
print("scale_pos_weight:", scale_pos_weight)

xgb_weighted = XGBClassifier(random_state=SEED, eval_metric='logloss', 
                               scale_pos_weight=scale_pos_weight)
xgb_weighted.fit(X_train_final, y_train)
y_pred_xgb_weighted = xgb_weighted.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb_weighted))

scale_pos_weight: 5.463958060288335
              precision    recall  f1-score   support

       False       0.94      0.91      0.93      2084
        True       0.59      0.71      0.65       382

    accuracy                           0.88      2466
   macro avg       0.77      0.81      0.79      2466
weighted avg       0.89      0.88      0.88      2466



## 10. Hyperparameter Tuning — Random Search (XGBoost)

In [14]:
grid = {
    'n_estimators': [300, 500, 700, 1000],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_lambda': [1, 3, 10, 20]
}

xgb_base = XGBClassifier(random_state=SEED, eval_metric='logloss',
                          scale_pos_weight=scale_pos_weight)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=grid,
    n_iter=10,
    scoring='f1',
    cv=4,
    random_state=SEED,
    n_jobs=-1
)

random_search.fit(X_train_final, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV F1-score:", random_search.best_score_)

Best parameters: {'subsample': 0.8, 'reg_lambda': 1, 'n_estimators': 300, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
Best CV F1-score: 0.6697947458741189


## 11. Hyperparameter Tuning — Grid Search (XGBoost)

In [15]:
grid_narrow = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.02, 0.03, 0.05],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_lambda': [1, 2, 5]
}

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=grid_narrow,
    scoring='f1',
    cv=4,
    n_jobs=-1
)

grid_search.fit(X_train_final, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", grid_search.best_score_)

Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.02, 'n_estimators': 400, 'reg_lambda': 1, 'subsample': 0.8}
Best CV F1-score: 0.6756449139205163


## 12a. Hyperparameter Tuning — Manual Search - Attempt 1

In [16]:
xgb_manual = XGBClassifier(
    n_estimators=400,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=1,
    random_state=SEED,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

s4f = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(xgb_manual, X_train_final, y_train, cv=s4f, scoring='f1')

print("CV F1-score:", cv_scores.mean())

CV F1-score: 0.6698478291616558


## 12b. Hyperparameter Tuning — Manual Search - Attempt 2

In [17]:
xgb_manual_v2 = XGBClassifier(
    n_estimators=500,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=1,
    random_state=SEED,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

cv_scores_v2 = cross_val_score(xgb_manual_v2, X_train_final, y_train, cv=s4f, scoring='f1')
print("CV F1-score (v2):", cv_scores_v2.mean())

CV F1-score (v2): 0.673641624065834


## 13. Final Model

In [18]:
final_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=1,
    random_state=SEED,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

final_model.fit(X_train_final, y_train)

y_pred_final = final_model.predict(X_test_final)
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

       False       0.96      0.89      0.92      2084
        True       0.57      0.82      0.67       382

    accuracy                           0.88      2466
   macro avg       0.77      0.85      0.80      2466
weighted avg       0.90      0.88      0.88      2466



In [19]:
# PageValues'sız versiyon
numerical_cols_no_pv = [col for col in numerical_cols if col != 'PageValues']

X_train_no_pv = pd.concat([X_train[numerical_cols_no_pv], X_train_cat_df], axis=1)
X_test_no_pv = pd.concat([X_test[numerical_cols_no_pv], X_test_cat_df], axis=1)

final_model_no_pv = XGBClassifier(
    n_estimators=400, learning_rate=0.02, subsample=0.8,
    colsample_bytree=0.7, reg_lambda=1, random_state=SEED,
    eval_metric='logloss', scale_pos_weight=scale_pos_weight
)
final_model_no_pv.fit(X_train_no_pv, y_train)

y_pred_no_pv = final_model_no_pv.predict(X_test_no_pv)
print(classification_report(y_test, y_pred_no_pv))

              precision    recall  f1-score   support

       False       0.93      0.70      0.80      2084
        True       0.30      0.71      0.42       382

    accuracy                           0.70      2466
   macro avg       0.62      0.70      0.61      2466
weighted avg       0.83      0.70      0.74      2466



## 13. Analysis & Results Summary

### Model Comparison

| Model | True Precision | True Recall | True F1-score |
|---|---|---|---|
| Dummy Classifier (baseline) | 0.00 | 0.00 | 0.00 |
| Random Forest (untuned) | 0.78 | 0.55 | 0.65 |
| XGBoost (untuned) | 0.73 | 0.59 | 0.65 |
| XGBoost (scale_pos_weight) | 0.59 | 0.71 | 0.65 |
| **XGBoost (tuned — final model)** | **0.57** | **0.82** | **0.67** |

The tuned XGBoost model achieved the highest F1-score for the minority class (`Revenue=True`), improving recall from 0.59 (untuned) to 0.82 while keeping F1 above all previous models. This means the final model correctly identifies **82% of actual purchasing sessions**, compared to only 59% before tuning — a substantial gain for a use case where missing a likely buyer is costly (e.g. retargeting campaigns).

The trade-off is a lower precision (0.57), meaning the model also flags more non-buyers as potential buyers. Given the business context (identifying sessions worth targeting for remarketing), prioritizing recall over precision is a reasonable choice.

### Tuning Process Impact

Three stages of hyperparameter tuning were applied, following the course methodology:

1. **Random Search** — CV F1-score: 0.6698
2. **Grid Search** (narrowed around Random Search results) — CV F1-score: 0.6756
3. **Manual Fine-Tuning** — CV F1-score: 0.6736 (did not surpass Grid Search)

The Grid Search result was selected as the final model configuration. The improvement across stages was modest (~0.6% F1), consistent with the expectation that tuning refines rather than transforms an already reasonable baseline.

### PageValues — Leakage Risk Finding

A critical finding emerged when testing the final model with and without the `PageValues` feature:

| | With PageValues | Without PageValues |
|---|---|---|
| True F1-score | **0.67** | **0.42** |

Removing `PageValues` caused a ~37% drop in F1-score, revealing that a large share of the model's predictive power depends on this single feature. This is consistent with the EDA finding that `PageValues` is the strongest correlate of `Revenue`, and raises a legitimate concern about when this feature is actually available (it likely finalizes late in a session).

**Decision:** Since this dataset represents completed sessions (post-session, not streaming data), the model is framed as a **post-session analysis tool** (e.g., identifying past sessions likely to convert, for retargeting or remarketing purposes) rather than a real-time prediction tool. Under this framing, using `PageValues` is valid and the final model (F1=0.67) is reported as the official result.

**Limitation:** If this model were to be deployed for real-time, mid-session prediction, the realistic expected performance would be closer to the PageValues-excluded result (F1≈0.42), since the finalized `PageValues` would not yet be available. This is noted as a direction for future work.

### Key Takeaways

- Class imbalance (~15.5% positive class) required careful metric selection; F1-score and confusion matrix were prioritized over accuracy, which was misleading (Dummy Classifier reached 85% accuracy while completely failing to identify buyers).
- Hyperparameter tuning improved recall substantially with only a minor precision trade-off, better aligning the model with the business goal of not missing likely buyers.
- `PageValues` is the dominant predictive feature, which is both the model's greatest strength and its main limitation — a finding that should be transparently communicated rather than hidden.